# Predicción del Valor de Mercado de Jugadores de Fútbol

**Instituto Tecnológico de Buenos Aires (ITBA)**  
**Materia:** Ciencia de Datos Aplicada — Licenciatura en Negocios y Tecnología  
**Alumnos:** Octavio Argonz y Matias Sola  

---

## Sección 0 — Objetivo del Proyecto

### ¿Qué predice el modelo?

El modelo estima el **valor de mercado futuro** (en euros) de jugadores de fútbol profesional. Dado el rendimiento de un jugador durante el año T (goles, asistencias, minutos jugados, posición, edad, liga) más su historial de valuaciones previas, el modelo predice cuánto debería valer en la **próxima ventana de transferencias** (año T+1).

Formalmente:
```
features del año T  →  predicción del valor en el año T+1
```

### ¿Por qué tiene valor de negocio?

Transfermarkt es el sitio de referencia mundial para valuaciones de jugadores. Hoy esas valuaciones las hace un equipo editorial de personas reales, de forma subjetiva y periódica. Eso genera ineficiencias: jugadores cuyo rendimiento justifica un valor mayor al asignado pueden ser comprados baratos por clubes con mejor análisis de datos (modelo Brentford/Brighton). Los beneficiarios directos son:

- **Clubes chicos** que buscan comprar subvaluados y venderlos con plusvalía.
- **Representantes** que necesitan argumentos cuantitativos para negociar contratos.
- **Fondos de inversión** que compran porcentajes de derechos económicos de jugadores jóvenes.

### ¿Qué cubre este notebook?

Este notebook corresponde al **Entregable 2**: Análisis Exploratorio de Datos (EDA) + preparación del dataset analítico. La construcción y evaluación de modelos de ML (Regresión Lineal/Ridge → Random Forest → XGBoost) se desarrollará en el Entregable 3.

### Estructura

| Sección | Contenido |
|---|---|
| 1 | Instalación e importación de librerías |
| 2 | Carga de datos |
| 3 | Análisis exploratorio básico (EDA) |
| 4 | Estadísticas de rendimiento por jugador-año |
| 5 | Construcción del target temporal |
| 6 | Dataset analítico unificado (`df_modelo`) |
| 7 | Feature engineering sin data leakage |
| 8 | Transformaciones |
| 9 | Split temporal y dataset final |
| 10 | Reflexión final |

## Sección 1 — Instalación e Importación de Librerías

Cargamos las librerías permitidas por la consigna y configuramos el estilo visual global. No se instalan dependencias externas fuera del stack estándar de Colab.

In [1]:
!pip install -q seaborn --upgrade

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14

print('Librerías cargadas correctamente.')
print(f'pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}')

Librerías cargadas correctamente.
pandas 2.2.2 | numpy 2.0.2 | seaborn 0.13.2


## Sección 2 — Carga de Datos

Montamos Google Drive y cargamos las cinco tablas del dataset *Football Data from Transfermarkt* (Kaggle, autor: davidcariboo). Los archivos deben estar en la carpeta `futbol-valuacion/data/raw/` dentro del Drive.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/futbol-valuacion/data/raw/'

Mounted at /content/drive


In [4]:
players      = pd.read_csv(DATA_PATH + 'players.csv')
appearances  = pd.read_csv(DATA_PATH + 'appearances.csv')
valuations   = pd.read_csv(DATA_PATH + 'player_valuations.csv')
transfers    = pd.read_csv(DATA_PATH + 'transfers.csv')
competitions = pd.read_csv(DATA_PATH + 'competitions.csv')

print('Tablas cargadas:')
for nombre, df in [('players', players), ('appearances', appearances),
                   ('player_valuations', valuations), ('transfers', transfers),
                   ('competitions', competitions)]:
    print(f'  {nombre}: {df.shape[0]:,} filas x {df.shape[1]} columnas')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/futbol-valuacion/data/raw/players.csv'

### `players` — Perfil de cada jugador

**Granularidad:** una fila = un jugador. Contiene datos estáticos del perfil (posición, fecha de nacimiento, nacionalidad, club actual) y una columna `market_value_in_eur` que es un **snapshot del momento de la última actualización** del dataset. Esta columna **no se usa como target** en el nuevo modelo (es el valor actual, no el futuro).

In [ ]:
print(f'Shape: {players.shape}')
print('\nColumnas y dtypes:')
print(players.dtypes)
display(players.head(3))

### `appearances` — Estadísticas por partido

**Granularidad:** una fila = un jugador en un partido específico. Contiene goles, asistencias, minutos jugados y tarjetas. Es la fuente principal de features de rendimiento: se agregará por `(player_id, año)` en la Sección 4.

In [ ]:
print(f'Shape: {appearances.shape}')
print('\nColumnas y dtypes:')
print(appearances.dtypes)
display(appearances.head(3))

### `player_valuations` — Historial de valuaciones

**Granularidad:** una fila = una valuación de un jugador en una fecha específica. Esta tabla es la fuente del **target**: el valor de mercado en el año T+1 se construye a partir de las entradas de esta tabla. También contiene `player_club_domestic_competition_id`, que identifica la liga del jugador **en el momento de cada valuación** (se usa en features para evitar data leakage).

In [ ]:
print(f'Shape: {valuations.shape}')
print('\nColumnas y dtypes:')
print(valuations.dtypes)
display(valuations.head(3))

### `transfers` — Historial de transferencias

**Granularidad:** una fila = una transferencia de un jugador. Contiene el fee pagado, clubes de origen y destino y fecha. Se incluye en el dataset pero no se usa como feature directa en este entregable.

In [ ]:
print(f'Shape: {transfers.shape}')
print('\nColumnas y dtypes:')
print(transfers.dtypes)
display(transfers.head(3))

### `competitions` — Catálogo de competiciones

**Granularidad:** una fila = una competición (liga o copa). Permite mapear `competition_id` a nombre de liga y país. Se usa como referencia para entender los valores de `player_club_domestic_competition_id`.

In [ ]:
print(f'Shape: {competitions.shape}')
print('\nColumnas y dtypes:')
print(competitions.dtypes)
display(competitions.head(3))

## Sección 3 — Análisis Exploratorio Básico (EDA)

Exploramos la estructura, distribuciones y calidad de los datos antes de transformarlos. El objetivo es entender el dataset para tomar decisiones informadas en las secciones siguientes.

### 3.1 — Granularidad y estructura relacional

Las tablas se vinculan principalmente a través de `player_id`. El esquema relacional es:

```
players (player_id) ←→ appearances (player_id, game_id)
players (player_id) ←→ player_valuations (player_id)
players (player_id) ←→ transfers (player_id)
appearances (competition_id) ←→ competitions (competition_id)
```

A continuación mostramos el rango temporal de `player_valuations` y la cobertura por año.

In [ ]:
valuations['date'] = pd.to_datetime(valuations['date'])
valuations['anio'] = valuations['date'].dt.year

print(f'Rango temporal de player_valuations:')
print(f'  Desde: {valuations["date"].min().date()}')
print(f'  Hasta: {valuations["date"].max().date()}')
print(f'  Años con datos: {valuations["anio"].min()} – {valuations["anio"].max()}')

registros_por_anio = valuations.groupby('anio').size().reset_index(name='registros')

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(registros_por_anio['anio'], registros_por_anio['registros'],
       color='steelblue', edgecolor='white')
ax.set_title('Registros de valuación por año')
ax.set_xlabel('Año')
ax.set_ylabel('Cantidad de registros')
plt.tight_layout()
plt.show()

print('\nRegistros por año (primeros y últimos 5):')
display(registros_por_anio)

### 3.2 — Distribuciones y outliers

Visualizamos la distribución del valor de mercado actual (de `players.csv`) para motivar la transformación logarítmica del target. También mostramos las distribuciones de las variables de rendimiento y el top 10 de jugadores por valor.

In [ ]:
mv = players['market_value_in_eur'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(mv / 1e6, bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Valor de mercado — escala original')
axes[0].set_xlabel('Valor (millones €)')
axes[0].set_ylabel('Cantidad de jugadores')

axes[1].hist(np.log1p(mv), bins=60, color='coral', edgecolor='white')
axes[1].set_title('Valor de mercado — escala log1p')
axes[1].set_xlabel('log(1 + Valor en €)')
axes[1].set_ylabel('Cantidad de jugadores')

plt.suptitle('Distribución del Valor de Mercado (snapshot actual)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Mediana: €{mv.median()/1e6:.2f}M | Media: €{mv.mean()/1e6:.2f}M | Máximo: €{mv.max()/1e6:.0f}M')
print(f'Skewness original: {mv.skew():.2f} → log1p: {np.log1p(mv).skew():.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, col, titulo in zip(axes,
                           ['goals', 'assists', 'minutes_played'],
                           ['Goles por partido', 'Asistencias por partido', 'Minutos jugados']):
    datos = appearances[col].dropna().clip(upper=appearances[col].quantile(0.99))
    ax.hist(datos, bins=50, color='mediumseagreen', edgecolor='white')
    ax.set_title(titulo)
    ax.set_xlabel(col)
    ax.set_ylabel('Apariciones')

plt.suptitle('Distribución de variables de rendimiento (appearances)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
top10 = (
    players[['name', 'market_value_in_eur', 'position']]
    .dropna(subset=['market_value_in_eur'])
    .sort_values('market_value_in_eur', ascending=False)
    .head(10)
    .copy()
)
top10['Valor'] = top10['market_value_in_eur'].apply(lambda x: f'€{x/1e6:.1f}M')
print('Top 10 jugadores por valor de mercado actual (snapshot):')
display(top10[['name', 'position', 'Valor']])

In [ ]:
Q1, Q3 = mv.quantile(0.25), mv.quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR
outliers = mv[mv > limite_superior]

print(f'Límite IQR superior: €{limite_superior/1e6:.2f}M')
print(f'Outliers detectados: {len(outliers):,} ({len(outliers)/len(mv)*100:.1f}% del total)')
print('→ No se eliminan: Mbappé, Haaland, etc. son valores reales, no errores.')

### 3.3 — Valores faltantes

Analizamos el porcentaje de nulos por columna en cada tabla. Es esperable que `market_value_in_eur` en `players` tenga ~35-40% de nulos: ligas menores con poca cobertura editorial en Transfermarkt.

In [ ]:
for nombre, df in [('players', players), ('appearances', appearances),
                   ('player_valuations', valuations), ('transfers', transfers)]:
    nulos = df.isnull().sum()
    nulos_pct = (nulos / len(df) * 100).round(2)
    resumen = pd.DataFrame({'Nulos': nulos, '% Nulos': nulos_pct}).query('Nulos > 0')
    if not resumen.empty:
        print(f'\n=== {nombre} ===')
        display(resumen.sort_values('% Nulos', ascending=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

for ax, (nombre, df) in zip(axes.flatten(),
    [('players', players), ('appearances', appearances),
     ('player_valuations', valuations), ('transfers', transfers)]):

    nulos_pct = df.isnull().mean() * 100
    nulos_pct = nulos_pct[nulos_pct > 0].sort_values(ascending=False)

    if nulos_pct.empty:
        ax.text(0.5, 0.5, 'Sin valores nulos', ha='center', va='center', fontsize=13)
        ax.set_title(nombre)
    else:
        nulos_pct.plot(kind='barh', ax=ax, color='tomato', edgecolor='white')
        ax.set_title(f'{nombre} — % Nulos por columna')
        ax.set_xlabel('% Nulos')
        ax.axvline(50, color='black', linestyle='--', linewidth=0.8)

plt.suptitle('Diagnóstico de Valores Faltantes', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.4 — Duplicados e inconsistencias

Verificamos duplicados en todas las tablas y detectamos jugadores sin apariciones registradas. Ambos son casos esperables: el dataset de Transfermarkt incluye jugadores de ligas menores con escasa cobertura.

In [ ]:
for nombre, df in [('players', players), ('appearances', appearances),
                   ('player_valuations', valuations), ('transfers', transfers)]:
    dupes = df.duplicated().sum()
    print(f'{nombre}: {dupes:,} filas duplicadas ({dupes/len(df)*100:.2f}%)')

In [ ]:
jugadores_con_apariciones = appearances['player_id'].nunique()
jugadores_total = players['player_id'].nunique()
sin_apariciones = jugadores_total - jugadores_con_apariciones

print(f'Jugadores únicos en players:              {jugadores_total:,}')
print(f'Con al menos una aparición registrada:    {jugadores_con_apariciones:,}')
print(f'Sin apariciones registradas:              {sin_apariciones:,} ({sin_apariciones/jugadores_total*100:.1f}%)')

ids_sin_match = set(valuations['player_id']) - set(players['player_id'])
print(f'\nIDs en valuations sin match en players:   {len(ids_sin_match):,}')

## Sección 4 — Estadísticas de Rendimiento por Jugador-Año

Este es el cambio central respecto al notebook anterior: en lugar de acumular las estadísticas de toda la carrera del jugador, **agregamos por `(player_id, año)`**. Esto permite construir features que representen el rendimiento del jugador en un año específico, sin mezclar temporalidades.

Extraemos el año del partido desde la columna `date` de `appearances`.

In [ ]:
appearances['date'] = pd.to_datetime(appearances['date'], errors='coerce')
appearances['anio'] = appearances['date'].dt.year

stats_por_anio = (
    appearances
    .dropna(subset=['anio'])
    .groupby(['player_id', 'anio'])
    .agg(
        n_apariciones     = ('game_id', 'count'),
        total_goles       = ('goals', 'sum'),
        total_asistencias = ('assists', 'sum'),
        total_minutos     = ('minutes_played', 'sum'),
    )
    .reset_index()
)

stats_por_anio['goles_por_partido']       = stats_por_anio['total_goles']        / stats_por_anio['n_apariciones']
stats_por_anio['asistencias_por_partido'] = stats_por_anio['total_asistencias']  / stats_por_anio['n_apariciones']
stats_por_anio['minutos_promedio']        = stats_por_anio['total_minutos']      / stats_por_anio['n_apariciones']

print(f'Shape stats_por_anio: {stats_por_anio.shape}')
print(f'Años disponibles: {stats_por_anio["anio"].min():.0f} – {stats_por_anio["anio"].max():.0f}')
display(stats_por_anio.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

apariciones_clip = stats_por_anio['n_apariciones'].clip(upper=60)
axes[0].hist(apariciones_clip, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Apariciones por jugador-año (recortadas en 60)')
axes[0].set_xlabel('Apariciones')
axes[0].set_ylabel('Cantidad de combinaciones jugador-año')

registros_anio = stats_por_anio.groupby('anio').size()
registros_anio.plot(kind='bar', ax=axes[1], color='mediumseagreen', edgecolor='white')
axes[1].set_title('Combinaciones jugador-año por año')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Combinaciones')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Estadísticas por Jugador-Año', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Sección 5 — Construcción del Target Temporal

El target del modelo es `market_value_next_year`: el valor de mercado del jugador en el año T+1. Se construye en tres pasos:

1. Para cada `(player_id, año)`, tomar la **última** valuación del año como `valor_en_anio`.
2. Hacer un **shift hacia adelante** por jugador para obtener el valor del año siguiente.
3. Eliminar filas donde `market_value_next_year` sea NaN (el último año de cada jugador no tiene futuro conocido).

Esto garantiza que el target sea genuinamente prospectivo: predecimos un valor que, en el momento de la predicción, aún no ocurrió.

In [ ]:
# Paso 1: última valuación por (jugador, año)
val_por_anio = (
    valuations
    .sort_values('date')
    .groupby(['player_id', 'anio'])['market_value_in_eur']
    .last()
    .reset_index()
    .rename(columns={'market_value_in_eur': 'valor_en_anio'})
)

# Paso 2: target = valor del año siguiente (shift -1 dentro de cada jugador)
val_por_anio = val_por_anio.sort_values(['player_id', 'anio'])
val_por_anio['market_value_next_year'] = (
    val_por_anio.groupby('player_id')['valor_en_anio'].shift(-1)
)

# Paso 3: eliminar filas sin target
valuaciones_con_target = val_por_anio.dropna(subset=['market_value_next_year']).copy()

print(f'Combinaciones jugador-año con target disponible: {len(valuaciones_con_target):,}')
print(f'Años con target: {valuaciones_con_target["anio"].min():.0f} – {valuaciones_con_target["anio"].max():.0f}')
display(valuaciones_con_target.head())

In [ ]:
target = valuaciones_con_target['market_value_next_year']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(target / 1e6, bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('market_value_next_year — escala original')
axes[0].set_xlabel('Valor (millones €)')
axes[0].set_ylabel('Registros')

axes[1].hist(np.log1p(target), bins=60, color='coral', edgecolor='white')
axes[1].set_title('market_value_next_year — escala log1p')
axes[1].set_xlabel('log(1 + Valor en €)')
axes[1].set_ylabel('Registros')

plt.suptitle('Distribución del Target Temporal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Mediana: €{target.median()/1e6:.2f}M | Media: €{target.mean()/1e6:.2f}M')
print(f'Skewness original: {target.skew():.2f} → log1p: {np.log1p(target).skew():.2f}')

## Sección 6 — Dataset Analítico Unificado (`df_modelo`)

Construimos `df_modelo` fusionando todas las fuentes. La granularidad final es **jugador-año**: una fila = un jugador en un año específico T, con features que representan lo que se sabía de ese jugador al cierre del año T y con target = su valor en T+1.

Pasos:
1. Join de `stats_por_anio` + `valuaciones_con_target` por `(player_id, anio)` — **inner join** (necesitamos tanto estadísticas como target).
2. Join con `players` para traer perfil (posición, fecha de nacimiento, etc.).
3. Calcular edad **en el año T** (no a la fecha actual).
4. Filtrar: retener solo filas con `n_apariciones >= 5`.

In [ ]:
# Paso 1: stats de rendimiento + target
df_modelo = stats_por_anio.merge(
    valuaciones_con_target[['player_id', 'anio', 'valor_en_anio', 'market_value_next_year']],
    on=['player_id', 'anio'],
    how='inner',
)
print(f'Después de join stats + target: {df_modelo.shape}')

# Paso 2: perfil del jugador
cols_players = ['player_id', 'position', 'sub_position', 'date_of_birth',
                'country_of_citizenship', 'current_club_domestic_competition_id']
cols_players = [c for c in cols_players if c in players.columns]

df_modelo = df_modelo.merge(players[cols_players], on='player_id', how='left')
print(f'Después de join con players:   {df_modelo.shape}')

# Paso 3: edad en el año T
df_modelo['date_of_birth'] = pd.to_datetime(df_modelo['date_of_birth'], errors='coerce')
df_modelo['edad_en_anio'] = df_modelo['anio'] - df_modelo['date_of_birth'].dt.year

# Paso 4: filtro por apariciones mínimas
MIN_APARICIONES = 5
antes = len(df_modelo)
df_modelo = df_modelo[df_modelo['n_apariciones'] >= MIN_APARICIONES].copy()
despues = len(df_modelo)
print(f'Después de filtro (>={MIN_APARICIONES} apariciones): {df_modelo.shape}')
print(f'Filas eliminadas: {antes - despues:,} ({(antes - despues)/antes*100:.1f}%)')

print(f'\nAños disponibles en df_modelo: {df_modelo["anio"].min():.0f} – {df_modelo["anio"].max():.0f}')
print('\nPrimeras filas:')
display(df_modelo.head())

In [ ]:
print('=== Resumen de df_modelo ===')
print(f'Filas (jugador-año):  {len(df_modelo):,}')
print(f'Columnas:             {df_modelo.shape[1]}')
print(f'Jugadores únicos:     {df_modelo["player_id"].nunique():,}')
print(f'Años:                 {sorted(df_modelo["anio"].unique())}')
print(f'\nColumnas:')
print(df_modelo.columns.tolist())

## Sección 7 — Feature Engineering sin Data Leakage

Construimos cuatro features derivadas. La regla central es que **para una fila con `anio = T`, ninguna feature puede usar información posterior al cierre del año T**. Esto garantiza que el modelo no ve el futuro durante el entrenamiento.

| Feature | Fuente | Descripción |
|---|---|---|
| `flag_club_grande` | `player_valuations` del año T | Jugador en las 5 grandes ligas europeas |
| `tendencia_valor` | `player_valuations` anteriores a T | Pendiente de las últimas 3 valuaciones (€/día) |
| `valor_mediano_liga` | `player_valuations` del año T | Mediana del valor en la liga del jugador en T |
| `nivel_relativo_mercado` | `player_valuations` del año T | Nivel global del mercado en T, base 2020 = 1.0 |

### 7.1 — `flag_club_grande`

Usamos `player_club_domestic_competition_id` de `player_valuations` en el año T (no `current_club_domestic_competition_id` de `players.csv`, que es el snapshot actual y podría referirse a un año diferente).

Vale `1` si el jugador estaba en GB1, ES1, L1, IT1 o FR1 en el año T.

In [ ]:
GRANDES_LIGAS = {'GB1', 'ES1', 'L1', 'IT1', 'FR1'}

# Liga del jugador en el año T: última valuación del año T
liga_en_anio = (
    valuations
    .dropna(subset=['player_club_domestic_competition_id'])
    .sort_values('date')
    .groupby(['player_id', 'anio'])['player_club_domestic_competition_id']
    .last()
    .reset_index()
    .rename(columns={'player_club_domestic_competition_id': 'liga_en_anio'})
)

df_modelo = df_modelo.merge(liga_en_anio, on=['player_id', 'anio'], how='left')
df_modelo['flag_club_grande'] = df_modelo['liga_en_anio'].isin(GRANDES_LIGAS).astype(int)

print(f'Jugadores en grandes ligas: {df_modelo["flag_club_grande"].sum():,}')
print(f'Jugadores en otras ligas:   {(df_modelo["flag_club_grande"]==0).sum():,}')
print(f'Sin liga identificada:      {df_modelo["liga_en_anio"].isna().sum():,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

counts = df_modelo['flag_club_grande'].value_counts().sort_index()
axes[0].bar(['Otras ligas', 'Gran liga'], counts.values, color=['steelblue', 'gold'], edgecolor='white')
axes[0].set_title('Distribución: Gran Liga vs. Otras')
axes[0].set_ylabel('Combinaciones jugador-año')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

df_flag = df_modelo[['flag_club_grande', 'market_value_next_year']].dropna().copy()
df_flag['liga'] = df_flag['flag_club_grande'].map({0: 'Otras ligas', 1: 'Gran liga'})
sns.boxplot(data=df_flag, x='liga', y='market_value_next_year',
            order=['Otras ligas', 'Gran liga'],
            palette={'Gran liga': 'gold', 'Otras ligas': 'steelblue'}, ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('Valor futuro por tipo de liga (escala log)')
axes[1].set_xlabel('')
axes[1].set_ylabel('market_value_next_year (€, log)')

plt.suptitle('flag_club_grande', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

corr_flag = df_modelo[['flag_club_grande', 'market_value_next_year']].dropna().corr().iloc[0, 1]
print(f'Correlación con log(market_value_next_year): r = {np.corrcoef(df_modelo["flag_club_grande"].dropna(), np.log1p(df_modelo["market_value_next_year"].dropna()))[0,1]:.3f}')

### 7.2 — `tendencia_valor`

Para cada `(player_id, T)`, tomamos las valuaciones con fecha **anterior al año T** (no del año T ni posteriores), tomamos las últimas 3, ajustamos una regresión lineal y extraemos la pendiente (€/día).

- Pendiente **positiva**: jugador en alza.
- Pendiente **negativa**: jugador en declive.
- `NaN`: menos de 2 valuaciones previas disponibles.

Usar solo valuaciones anteriores a T garantiza que no se mira el futuro.

In [ ]:
valuations['date_ord'] = valuations['date'].map(pd.Timestamp.toordinal)

def pendiente_previa(player_id, anio_t, val_df):
    previas = val_df[(val_df['player_id'] == player_id) & (val_df['anio'] < anio_t)]
    ult = previas.sort_values('date').tail(3)
    if len(ult) < 2:
        return np.nan
    x = ult['date_ord'].values.astype(float)
    y = ult['market_value_in_eur'].values.astype(float)
    coefs = np.polyfit(x - x.mean(), y, 1)
    return coefs[0]

# Precalculamos para cada combinación jugador-año única en df_modelo
print('Calculando tendencias (puede tardar unos minutos)...')
pares = df_modelo[['player_id', 'anio']].drop_duplicates()

# Optimización: indexamos valuaciones por player_id
val_indexed = valuations.set_index('player_id')

resultados = []
for _, row in pares.iterrows():
    pid, anio_t = row['player_id'], row['anio']
    try:
        subset = val_indexed.loc[[pid]].reset_index() if pid in val_indexed.index else pd.DataFrame()
    except KeyError:
        subset = pd.DataFrame()
    if subset.empty:
        resultados.append({'player_id': pid, 'anio': anio_t, 'tendencia_valor': np.nan})
        continue
    previas = subset[subset['anio'] < anio_t].sort_values('date').tail(3)
    if len(previas) < 2:
        resultados.append({'player_id': pid, 'anio': anio_t, 'tendencia_valor': np.nan})
        continue
    x = previas['date_ord'].values.astype(float)
    y = previas['market_value_in_eur'].values.astype(float)
    coefs = np.polyfit(x - x.mean(), y, 1)
    resultados.append({'player_id': pid, 'anio': anio_t, 'tendencia_valor': coefs[0]})

tendencias_df = pd.DataFrame(resultados)
df_modelo = df_modelo.merge(tendencias_df, on=['player_id', 'anio'], how='left')

cobertura = df_modelo['tendencia_valor'].notna().mean() * 100
print(f'Cobertura de tendencia_valor: {cobertura:.1f}%')
display(df_modelo['tendencia_valor'].describe())

In [ ]:
p1, p99 = df_modelo['tendencia_valor'].quantile([0.01, 0.99])
tv_clip = df_modelo['tendencia_valor'].clip(p1, p99)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(tv_clip.dropna() / 1000, bins=60, color='mediumpurple', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Tendencia = 0')
axes[0].set_title('Distribución de tendencia_valor (p1-p99)')
axes[0].set_xlabel('Pendiente (miles de €/día)')
axes[0].set_ylabel('Combinaciones jugador-año')
axes[0].legend()

df_sc = df_modelo[['tendencia_valor', 'market_value_next_year']].dropna()
df_sc = df_sc[df_sc['tendencia_valor'].between(p1, p99)]
axes[1].scatter(df_sc['tendencia_valor'] / 1000, np.log1p(df_sc['market_value_next_year']),
                alpha=0.15, s=6, color='mediumpurple')
axes[1].set_title('Tendencia vs. log(market_value_next_year)')
axes[1].set_xlabel('Pendiente (miles de €/día)')
axes[1].set_ylabel('log(1 + valor futuro)')

plt.suptitle('tendencia_valor', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

corr_tv = np.corrcoef(
    df_sc['tendencia_valor'],
    np.log1p(df_sc['market_value_next_year'])
)[0, 1]
print(f'Correlación con log(market_value_next_year): r = {corr_tv:.3f}')

### 7.3 — `valor_mediano_liga`

Para cada `(player_id, T)`, calculamos la mediana de `market_value_in_eur` de todos los jugadores en la misma liga en el año T. Usamos la liga del jugador en el año T (obtenida en 7.1) para evitar data leakage.

Esta variable captura el *nivel económico del entorno*: el mismo rendimiento vale diferente en la Premier League que en una liga menor.

In [ ]:
# Mediana de valor por (liga, año) usando player_valuations
mediana_liga_anio = (
    valuations
    .dropna(subset=['player_club_domestic_competition_id', 'market_value_in_eur'])
    .groupby(['player_club_domestic_competition_id', 'anio'])['market_value_in_eur']
    .median()
    .reset_index()
    .rename(columns={
        'market_value_in_eur': 'valor_mediano_liga',
        'player_club_domestic_competition_id': 'liga_join',
    })
)

df_modelo = df_modelo.merge(
    mediana_liga_anio,
    left_on=['liga_en_anio', 'anio'],
    right_on=['liga_join', 'anio'],
    how='left',
).drop(columns=['liga_join'], errors='ignore')

cobertura_liga = df_modelo['valor_mediano_liga'].notna().mean() * 100
print(f'Cobertura de valor_mediano_liga: {cobertura_liga:.1f}%')
display(df_modelo['valor_mediano_liga'].describe())

In [ ]:
anio_max = int(df_modelo['anio'].max())
top_ligas_df = (
    mediana_liga_anio[mediana_liga_anio['anio'] == anio_max]
    .sort_values('valor_mediano_liga', ascending=False)
    .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(
    top_ligas_df['liga_join'][::-1],
    top_ligas_df['valor_mediano_liga'][::-1] / 1e6,
    color='darkorange', edgecolor='white',
)
axes[0].set_title(f'Top 15 ligas — Mediana de valor ({anio_max})')
axes[0].set_xlabel('Mediana (millones €)')

for liga in ['GB1', 'ES1', 'L1', 'IT1', 'FR1']:
    datos = mediana_liga_anio[mediana_liga_anio['liga_join'] == liga]
    if not datos.empty:
        axes[1].plot(datos['anio'], datos['valor_mediano_liga'] / 1e6,
                     marker='o', label=liga)
axes[1].set_title('Evolución del valor mediano — 5 grandes ligas')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Mediana (millones €)')
axes[1].legend()

plt.suptitle('valor_mediano_liga', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

df_corr_liga = df_modelo[['valor_mediano_liga', 'market_value_next_year']].dropna()
r_liga = np.corrcoef(
    df_corr_liga['valor_mediano_liga'],
    np.log1p(df_corr_liga['market_value_next_year'])
)[0, 1]
print(f'Correlación con log(market_value_next_year): r = {r_liga:.3f}')

### 7.4 — `nivel_relativo_mercado`

Mediana global de valuaciones del año T, normalizada sobre el año 2020 como base = 1.0. Mide el nivel interno del mercado de Transfermarkt en el año T.

**Importante:** el nombre anterior (`indice_inflacion_mercado`) era confuso porque sugería inflación económica real. Este índice no es el IPC ni usa fees de transferencias reales — es simplemente la mediana global de valuaciones editoriales de Transfermarkt del año T, normalizada sobre 2020. El nombre `nivel_relativo_mercado` lo describe con precisión.

Un valor > 1 indica que el mercado estaba inflado respecto a 2020; < 1 indica deflación (ej. caída en 2020 por pandemia).

In [ ]:
mediana_global_por_anio = (
    valuations
    .dropna(subset=['market_value_in_eur'])
    .groupby('anio')['market_value_in_eur']
    .median()
)

if 2020 in mediana_global_por_anio.index:
    base_2020 = mediana_global_por_anio[2020]
else:
    idx_cercano = (mediana_global_por_anio.index - 2020).abs().argmin()
    base_2020 = mediana_global_por_anio.iloc[idx_cercano]
    print(f'Año 2020 no disponible; usando base = {base_2020:.0f} €')

nivel_relativo = (mediana_global_por_anio / base_2020).reset_index()
nivel_relativo.columns = ['anio', 'nivel_relativo_mercado']

print('Nivel relativo del mercado por año (base 2020 = 1.0):')
display(nivel_relativo)

df_modelo = df_modelo.merge(nivel_relativo, on='anio', how='left')
print(f'\nCobertura de nivel_relativo_mercado: {df_modelo["nivel_relativo_mercado"].notna().mean()*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(nivel_relativo['anio'], nivel_relativo['nivel_relativo_mercado'],
             marker='o', color='darkcyan', linewidth=2)
axes[0].axhline(1.0, color='red', linestyle='--', linewidth=1.2, label='Base 2020')
axes[0].fill_between(
    nivel_relativo['anio'], nivel_relativo['nivel_relativo_mercado'], 1.0,
    where=(nivel_relativo['nivel_relativo_mercado'] >= 1.0),
    alpha=0.2, color='green', label='Por encima de 2020'
)
axes[0].fill_between(
    nivel_relativo['anio'], nivel_relativo['nivel_relativo_mercado'], 1.0,
    where=(nivel_relativo['nivel_relativo_mercado'] < 1.0),
    alpha=0.2, color='red', label='Por debajo de 2020'
)
axes[0].set_title('Evolución del nivel relativo de mercado')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Nivel relativo (2020 = 1.0)')
axes[0].legend(fontsize=9)

axes[1].hist(df_modelo['nivel_relativo_mercado'].dropna(),
             bins=20, color='darkcyan', edgecolor='white')
axes[1].set_title('Distribución del nivel relativo en df_modelo')
axes[1].set_xlabel('Nivel relativo (2020 = 1.0)')
axes[1].set_ylabel('Combinaciones jugador-año')

plt.suptitle('nivel_relativo_mercado', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

df_corr_nrm = df_modelo[['nivel_relativo_mercado', 'market_value_next_year']].dropna()
r_nrm = np.corrcoef(
    df_corr_nrm['nivel_relativo_mercado'],
    np.log1p(df_corr_nrm['market_value_next_year'])
)[0, 1]
print(f'Correlación con log(market_value_next_year): r = {r_nrm:.3f}')

## Sección 8 — Transformaciones

Aplicamos las tres transformaciones necesarias para preparar el dataset para el modelado:

1. **Log del target**: corregir sesgo en `market_value_next_year`.
2. **Encoding ordinal de posición**: convertir la posición a un número.
3. **StandardScaler**: estandarizar variables numéricas continuas.

### 8.1 — Transformación Logarítmica del Target

In [ ]:
df_modelo['log_target'] = np.log1p(df_modelo['market_value_next_year'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_modelo['market_value_next_year'].dropna() / 1e6, bins=60,
             color='steelblue', edgecolor='white')
axes[0].set_title('market_value_next_year — escala original')
axes[0].set_xlabel('Valor (millones €)')
axes[0].set_ylabel('Registros')

axes[1].hist(df_modelo['log_target'].dropna(), bins=60, color='coral', edgecolor='white')
axes[1].set_title('log_target = log1p(market_value_next_year)')
axes[1].set_xlabel('log(1 + Valor en €)')
axes[1].set_ylabel('Registros')

plt.suptitle('Efecto de la Transformación Logarítmica sobre el Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Skewness original: {df_modelo["market_value_next_year"].skew():.2f}')
print(f'Skewness log:      {df_modelo["log_target"].skew():.2f}')

### 8.2 — Encoding de Posición

Mapeo ordinal: Goalkeeper=1, Defender=2, Midfielder=3, Forward=4. Preferimos `sub_position` (más específico) con fallback a `position`.

In [ ]:
ENCODING_POSICION = {
    'Goalkeeper': 1,
    'Defender': 2, 'Centre-Back': 2, 'Left-Back': 2, 'Right-Back': 2, 'Defence': 2,
    'Midfielder': 3, 'Midfield': 3, 'Central Midfield': 3, 'Defensive Midfield': 3,
    'Attacking Midfield': 3, 'Left Midfield': 3, 'Right Midfield': 3,
    'Forward': 4, 'Centre-Forward': 4, 'Left Winger': 4, 'Right Winger': 4,
    'Second Striker': 4, 'Attack': 4,
}

pos_col = 'sub_position' if 'sub_position' in df_modelo.columns else 'position'
print(f'Columna usada: {pos_col}')
print('Valores únicos encontrados:')
print(sorted(df_modelo[pos_col].dropna().unique()))

df_modelo['posicion_num'] = df_modelo[pos_col].map(ENCODING_POSICION)

# Fallback a position si sub_position dejó NaN
if pos_col == 'sub_position' and 'position' in df_modelo.columns:
    faltantes = df_modelo['posicion_num'].isna()
    df_modelo.loc[faltantes, 'posicion_num'] = (
        df_modelo.loc[faltantes, 'position'].map(ENCODING_POSICION)
    )

mapeados = df_modelo['posicion_num'].notna().sum()
print(f'\nJugador-años con posición mapeada: {mapeados:,} / {len(df_modelo):,} ({mapeados/len(df_modelo)*100:.1f}%)')

print('\nDistribución por código de posición:')
display(df_modelo['posicion_num'].value_counts().sort_index())

### 8.3 — Normalización con StandardScaler

Estandarizamos las variables numéricas continuas (media = 0, desvío = 1). Imputamos NaN por mediana antes de escalar. Esta es la única estandarización del pipeline.

In [ ]:
VARS_CONTINUAS = [
    'edad_en_anio', 'goles_por_partido', 'asistencias_por_partido',
    'minutos_promedio', 'tendencia_valor', 'valor_mediano_liga', 'nivel_relativo_mercado',
]
VARS_CONTINUAS = [c for c in VARS_CONTINUAS if c in df_modelo.columns]

print('Variables a estandarizar:')
print(VARS_CONTINUAS)

df_scaler_input = df_modelo[VARS_CONTINUAS].copy()
df_scaler_input = df_scaler_input.fillna(df_scaler_input.median())

scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_scaler_input),
    columns=[f'{c}_scaled' for c in VARS_CONTINUAS],
    index=df_modelo.index,
)

for col in df_scaled.columns:
    df_modelo[col] = df_scaled[col]

print('\n=== ANTES (escala original) ===')
display(df_scaler_input.describe().round(2).T)

print('\n=== DESPUÉS (media≈0, desvío≈1) ===')
display(df_scaled.describe().round(2).T)

## Sección 9 — Split Temporal y Dataset Final

Dividimos `df_modelo` en entrenamiento y test usando el año como criterio:

- **Train**: años ≤ 2022 (datos históricos)
- **Test**: años ≥ 2023 (datos más recientes)

El split es **temporal** (no aleatorio) para evitar data leakage temporal: el modelo nunca puede ver datos futuros durante el entrenamiento. Un split aleatorio mezclaría años y permitiría al modelo aprender el valor de 2023 del mismo jugador mientras entrena con sus datos de 2022.

In [ ]:
train = df_modelo[df_modelo['anio'] <= 2022].copy()
test  = df_modelo[df_modelo['anio'] >= 2023].copy()

print(f'Train: {len(train):,} filas ({len(train)/len(df_modelo)*100:.1f}%) — años ≤ 2022')
print(f'Test:  {len(test):,} filas ({len(test)/len(df_modelo)*100:.1f}%) — años ≥ 2023')
print(f'\nAños en train: {sorted(train["anio"].unique())}')
print(f'Años en test:  {sorted(test["anio"].unique())}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train['log_target'].dropna(), bins=40, alpha=0.7, color='steelblue',
             edgecolor='white', label='Train')
axes[0].hist(test['log_target'].dropna(), bins=40, alpha=0.7, color='coral',
             edgecolor='white', label='Test')
axes[0].set_title('Distribución del log_target por split')
axes[0].set_xlabel('log(1 + market_value_next_year)')
axes[0].set_ylabel('Registros')
axes[0].legend()

cant_por_anio = df_modelo.groupby('anio').size()
colores = ['steelblue' if a <= 2022 else 'coral' for a in cant_por_anio.index]
cant_por_anio.plot(kind='bar', ax=axes[1], color=colores, edgecolor='white')
axes[1].set_title('Combinaciones jugador-año por año (azul=train, rojo=test)')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Combinaciones')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Split Temporal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
FEATURES = [
    'edad_en_anio', 'posicion_num',
    'goles_por_partido', 'asistencias_por_partido', 'minutos_promedio',
    'flag_club_grande', 'tendencia_valor',
    'valor_mediano_liga', 'nivel_relativo_mercado',
]
TARGET = 'log_target'

print('FEATURES:')
for f in FEATURES:
    disponible = f in df_modelo.columns
    cobertura = df_modelo[f].notna().mean() * 100 if disponible else 0
    print(f'  {f}: {"OK" if disponible else "FALTA"} — {cobertura:.1f}% no-nulo')

print(f'\nTARGET: {TARGET}')
print(f'  {df_modelo[TARGET].notna().mean()*100:.1f}% no-nulo')

print('\nEstadísticas descriptivas del dataset final:')
display(df_modelo[FEATURES + [TARGET]].describe())

In [ ]:
features_disponibles = [f for f in FEATURES if f in df_modelo.columns]

corr_target = (
    df_modelo[features_disponibles + [TARGET]]
    .corr()[[TARGET]]
    .drop(TARGET)
    .sort_values(TARGET, ascending=False)
)
print('Correlación de cada feature con log_target:')
display(corr_target)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    df_modelo[features_disponibles + [TARGET]].corr(),
    annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax,
)
ax.set_title('Matriz de correlaciones — features y target')
plt.tight_layout()
plt.show()

## Sección 10 — Reflexión Final

### Decisiones tomadas y justificación

**Granularidad jugador-año.**  
La versión anterior del notebook usaba una fila por jugador con estadísticas acumuladas de toda la carrera, lo que generaba mezcla temporal (goles de 2015 junto al valor de 2024). La nueva granularidad jugador-año elimina ese problema: cada fila representa el estado del jugador en un año específico T, y el target es el valor en T+1. Esto permite entrenar con ejemplos históricos reales y aplicar el modelo con datos del año corriente para predecir el siguiente.

**Inner join en la Sección 6.**  
Se eligió inner join entre las estadísticas de `appearances` y `player_valuations`. Un left join preservaría jugadores sin valuación en ese año, generando NaN en el target. No se puede entrenar un modelo de regresión con filas sin target.

**Filtro de mínimo 5 apariciones por año.**  
Con una sola aparición, el promedio de goles es poco representativo (un gol en un partido da `goles_por_partido = 1`). El umbral de 5 apariciones es un compromiso entre representatividad estadística y retención de datos.

**Transformación logarítmica del target.**  
El valor de mercado tiene distribución fuertemente sesgada. `log1p` acerca la distribución a la normalidad, beneficia a modelos lineales y reduce la influencia de outliers sin eliminarlos. Los outliers (Mbappé, Haaland) son valores reales, no errores.

**Encoding ordinal de posición.**  
El orden Goalkeeper=1 → Forward=4 aproxima la zonificación del campo y la diferencia esperada en valor de mercado por posición. Para el Entregable 3 se pueden probar también las dummies one-hot para modelos no lineales.

**Feature engineering sin data leakage.**  
Cada feature del año T usa exclusivamente información disponible hasta el cierre de T:  
- `flag_club_grande`: liga del jugador en el año T (de `player_valuations`, no de `players.csv` que es snapshot actual).  
- `tendencia_valor`: pendiente calculada con valuaciones anteriores a T.  
- `valor_mediano_liga`: mediana de la liga en el año T.  
- `nivel_relativo_mercado`: mediana global del año T normalizada sobre 2020.

**Split temporal.**  
Train ≤ 2022, test ≥ 2023. Un split aleatorio mezclaría años y permitiría al modelo ver el valor de 2023 de un jugador mientras entrena con sus datos de 2022 — eso es data leakage temporal. El split por año garantiza que el set de evaluación simula una predicción genuinamente futura.

**Renombrado de `nivel_relativo_mercado`.**  
El nombre anterior (`indice_inflacion_mercado`) era incorrecto: no es inflación económica real (IPC), no usa fees de transferencias reales. Es simplemente la mediana global de valuaciones editoriales de Transfermarkt del año T, normalizada sobre 2020. El nuevo nombre lo describe sin confusión.

---

### Dificultades encontradas

- **Problema conceptual en el notebook anterior**: el dataset tenía una fila por jugador con estadísticas acumuladas, lo que mezclaba temporalmente features y target. Corregirlo requirió replantear toda la arquitectura del pipeline.
- **Cálculo de `tendencia_valor` sin leakage**: calcular la pendiente usando solo valuaciones anteriores a T requirió una iteración por cada par `(player_id, anio)`. Es computacionalmente intensivo pero necesario para garantizar la integridad del pipeline.
- **Cobertura desigual de `player_valuations`**: hay más registros en años recientes. Jugadores de ligas menores o con pocas entradas en Transfermarkt tienen NaN en el target para muchos años.
- **Múltiples valores en `position` / `sub_position`**: Transfermarkt tiene categorías inconsistentes entre idiomas y actualizaciones. El encoding cubre las principales, con fallback para las raras.

---

### Próximos pasos — Entregable 3

1. **Modelos baseline**: Regresión Lineal y Ridge sobre `log_target` con las features estandarizadas.
2. **Modelo intermedio**: Random Forest para capturar interacciones no lineales (edad × posición, liga × rendimiento).
3. **Modelo avanzado**: XGBoost como referente de boosting — se espera el mejor desempeño.
4. **Métricas**: RMSE y MAE en escala original (€) para interpretabilidad, R² sobre `log_target` para comparación entre modelos. Objetivo: R² ≥ 0.70.
5. **Validación cruzada temporal**: `TimeSeriesSplit` para estimar robustez.
6. **Interpretabilidad**: SHAP values sobre el mejor modelo para entender el peso real de cada feature.